In [0]:
import os

csv_dir = '/mnt/bd/openuniverselake/aisraw/aisdata/csv'
delta_dir = csv_dir.replace('/csv', '/delta')
csv_view = "CREATE OR REPLACE TEMPORARY VIEW ais_csv USING csv OPTIONS ( path '**CSV**',   header 'true' );"
raw_view = """CREATE OR REPLACE TEMPORARY VIEW ais_raw 
                AS
                SELECT DISTINCT `# Timestamp` rec_time, `Type of mobile` mobile_type, MMSI  mmsi, Latitude latitude, Longitude longitude, `Navigational status` nav_status, ROT rot, SOG sog, COG cog, Heading heading, IMO imo, Callsign callsign, Name name
                        , `Ship type` ship_type, `Cargo type` cargo_type, Width ship_width, Length ship_length, `Type of position fixing device` device, Draught draught, Destination destination, ETA eta, `Data source type` source_type, A a, B b, C c,  D d 
                FROM ais_csv;"""
ais_view = """CREATE OR REPLACE TEMPORARY VIEW ais
                AS
                SELECT to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss') as rec_time
                    , mobile_type, mmsi
                    , cast(latitude as NUMERIC(9,6)) AS latitude , CAST( longitude as NUMERIC(9,6)) AS longitude
                    , nav_status
                    , CAST( rot as double) AS rot
                    , CAST( sog as double) AS sog
                    , CAST( cog as double) AS cog
                    , CAST( heading as double) AS heading
                    , imo, callsign, name, ship_type, cargo_type
                    , CAST( ship_width as double) AS ship_width
                    , CAST( ship_length as double) AS ship_length
                    , device
                    , CAST( draught as double) AS draught
                    , destination,  eta, source_type
                    , CAST( a as double) AS a
                    , CAST( b as double) AS b
                    , CAST( c as double) AS c
                    , CAST( d as double) AS d 
                    , year(to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss')) as yyyy
                    , date_format(to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss'), 'yyyy-MM') as yyyy_mm
                    , date_format(to_timestamp(rec_time, 'dd/MM/yyyy HH:mm:ss'), 'yyyy-MM-dd') as yyyy_mm_dd
                FROM ais_raw; """

csv_files = [os.path.join(csv_dir, f) for f in os.listdir(f"/dbfs/{csv_dir}") if os.path.isfile(os.path.join(f"/dbfs/{csv_dir}", f))]

print(f"Target directory {delta_dir}")
for f in csv_files:
    replace_where = f"`yyyy_mm_dd` == '{f.replace(csv_dir, '').replace('/', '').replace('.csv', '').replace('aisdk-', '')}'"  # https://stackoverflow.com/questions/70261756/databricks-overwriting-entire-table-instead-of-adding-new-partition
    print(f"Convert {f} ({replace_where})...")
    sql_view = csv_view.replace('**CSV**', f)
    spark.sql(sql_view)
    spark.sql(raw_view)
    spark.sql(ais_view)
    df = spark.sql("select * from ais")
    df.coalesce(1).write.partitionBy("yyyy", "yyyy_mm", "yyyy_mm_dd").option("maxRecordsPerFile", 20000000).format("delta").mode("overwrite").option("replaceWhere", replace_where).save(delta_dir)

Post fix, if DISTINCT above forgotten
INSERT OVERWRITE TABLE ${schema_name}.messages
PARTITION (yyyy, yyyy_mm, yyyy_mm_dd)
SELECT DISTINCT * FROM ${schema_name}.messages;